### 과제2-1

접근 : 
- 그림에 주어진 표를 이용해 행렬로 list를 만든 후 접근하기

1. 목적함수 :  
min(2 x[0][0] +3 x[0][1] +5 x[0][2] +2.5 x[1][0] +4 x[1][1] +4.8 x[1][2] +3 x[2][0] +3.6 x[2][1] +3.2 x[2][2])

2. 제약식 :   
- 가구회사 수요: 
    - sum(x[0][n]) >= 500 
    - sum(x[1][n]) >= 700
    - sum(x[2][n]) >= 600 
- 목재3 공급제한: 
    - sum(x[2][n])<= 500  
- 가구 운송 제약: 
    - x[1][0], x[1][1],... ,x[2][2]까지 각각 <=200 

In [3]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SCIP")

# 비용($ per ton)
C = [
    [2, 3, 5],
    [2.5, 4, 4.8],
    [3, 3.6, 3.2]
]

# 주당 목재 수요
D = [500, 700, 600]

n = 3

# 의사결정변수
# y[i,j] = 목재 회사 i가 가구 공장 j에게
x = {}
for i in range(n):
    for j in range(n):
        x[i, j] = solver.IntVar(0, solver.infinity(), "x[%i][%i]" % (i, j))

# 제약조건
# 가구의 주당 수요
for j in range(n):
    solver.Add(sum(x[i, j] for i in range(n)) == D[j], "가구 회사 수요")

# 목재3 공급 제한
solver.Add(sum(x[2, j] for j in range(n)) <= 500, "목재3 공급 제한")

# 가구별 운송 제약
for i in range(1,3):
    for j in range(n):
        solver.Add(x[i, j] <= 200, "가구 운송 제약")

obj_expr = []
for i in range(n):
    for j in range(n):
        obj_expr.append(C[i][j] * x[i, j])

solver.Minimize(solver.Sum(obj_expr))

# 모델 파일 생성
with open("or_과제2_1.lp", "w") as out_f:
    lp_text = solver.ExportModelAsLpFormat(False)
    out_f.write(lp_text)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("OPTIMAL")
    print("목적함수값 =", solver.Objective().Value(), "($)")

    print("\n[목재별 가구회사들의 구매량(ton)]")
    print("      ", end="")
    for j in range(n):
        print("가구[%i]\t" % (j + 1), end="")
    print()

    for i in range(n):
        print("목재[%i]\t" % (i + 1), end="")
        for j in range(n):
            print("%.1f\t" % x[i, j].solution_value(), end="")
        print()

else:
    print("The problem does not have an optimal solution.")

OPTIMAL
목적함수값 = 5700.0 ($)

[목재별 가구회사들의 구매량(ton)]
      가구[1]	가구[2]	가구[3]	
목재[1]	500.0	700.0	200.0	
목재[2]	0.0	0.0	200.0	
목재[3]	0.0	0.0	200.0	


### 과제2-2

용어   
- acre: 운동장 절반 크기
- unit : 가구수(=주택수)

조건
- 제한 10 acre
- low : 비용 20 units/acre * 13000$/unit, 제한 60~100가구
- middle : 비용 15 units/acre * 18000$/unit, 제한 30~70가구
- 도합 가구 최대 150, 담보 최대 200만$
- (low units)>=50+1/2*(middle units)
  
접근
- 한 에이커에 두 종류의 주택이 들어설 수 있다는 걸 감안하려면,  
한 에이커당 주택을 주택당 에이커로 바꾸어서 밀도로 접근해야한다.

a. 비용 최소

1. 변수 정의 : 
- a,b : low&middle 가구수

2. 목적함수 : 
min(13000*a+18000*b)
  
3. 제약식 : 
- 1/20*a+1/15*b<=10
- 13000*a+18000*b<=2000000
- 60<=a<=100
- 30<=b<=70
- a+b<=150
- a>=50+b/2

In [8]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SAT")

infinity = solver.infinity()

x = solver.IntVar(0.0,infinity, "x")
y = solver.IntVar(0.0,infinity, "y")

solver.Add(1/20*x + 1/15*y <=10)
solver.Add(13000*x + 18000*y <=2000000)
solver.Add(60<= x)
solver.Add(x <=100)
solver.Add(30<= y)
solver.Add(y <=70)
solver.Add(x + y <= 150)
solver.Add(50 + y/2 <= x)

solver.Minimize(13000*x + 18000*y)

print(f"Solving with {solver.SolverVersion()}")
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Solution:")
    print("Objective value(비용)=", solver.Objective().Value(), "($)")
    print("low-income-house 수 : ", x.solution_value(), "(가구)")
    print("middle-income-house 수 : ", y.solution_value(), "(가구)")
else:
    print("The problem does not have an optimal solution.")


Solving with CP-SAT solver v9.15.6755
Solution:
Objective value(비용)= 1385000.0 ($)
low-income-house 수 :  65.0 (가구)
middle-income-house 수 :  30.0 (가구)


b. 주택수 최대

1. 변수 정의 : 
- a,b : low&middle 가구수

2. 목적함수 : 
max(a+b)
  
3. 제약식 : 
- 1/20*a+1/15*b<=10
- 13000*a+18000*b<=2000000
- 60<=a<=100
- 30<=b<=70
- a+b<=150
- a>=50+b/2

In [9]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("SAT")

infinity = solver.infinity()

x = solver.IntVar(0.0,infinity, "x")
y = solver.IntVar(0.0,infinity, "y")

solver.Add(1/20*x + 1/15*y <=10)
solver.Add(13000*x + 18000*y <=2000000)
solver.Add(60<= x)
solver.Add(x <=100)
solver.Add(30<= y)
solver.Add(y <=70)
solver.Add(x + y <= 150)
solver.Add(50 + y/2 <= x)

solver.Maximize(x + y)

print(f"Solving with {solver.SolverVersion()}")
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Solution:")
    print("Objective value(총 주택수)", solver.Objective().Value(), "(가구)")
    print("low-income-house 수 : ", x.solution_value(), "(가구)")
    print("middle-income-house 수 : ", y.solution_value(), "(가구)")
else:
    print("The problem does not have an optimal solution.")


Solving with CP-SAT solver v9.15.6755
Solution:
Objective value(총 주택수) 138.0 (가구)
low-income-house 수 :  100.0 (가구)
middle-income-house 수 :  38.0 (가구)
